# DeepSketch Phase 2 — Kaggle Training Notebook
Train Pix2Pix for `sketch -> stylized color portrait`.
This notebook orchestrates repository scripts only; no model/training logic is duplicated here.

## 1. Check Kaggle GPU & Setup Environment

In [ ]:
import os
import torch

os.environ['WANDB_MODE'] = 'disabled'
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Clone the GitHub Repository

In [ ]:
!rm -rf deep-sketch
!git clone https://github.com/supratimcoder1/deep-sketch.git
%cd deep-sketch

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt -q

## 4. Verify Kaggle Dataset Location

In [ ]:
from pathlib import Path
import os

dataset_root = Path('/kaggle/input/datasets/supratimghosh01/cufs-dataset-clean-stylized/dataset')
photos_src = dataset_root / 'photos'
sketches_src = dataset_root / 'sketches'
stylized_src = dataset_root / 'stylized'

print('photos exists:', photos_src.exists())
print('sketches exists:', sketches_src.exists())
print('stylized exists:', stylized_src.exists())

if not photos_src.exists() or not sketches_src.exists() or not stylized_src.exists():
    raise FileNotFoundError('Expected /kaggle/input/datasets/supratimghosh01/cufs-dataset-clean-stylized/dataset/{photos,sketches,stylized}')

print('Photos found:', len(os.listdir(photos_src)))
print('Sketches found:', len(os.listdir(sketches_src)))
print('Stylized found:', len(os.listdir(stylized_src)))

## 5. Create Dataset Symlinks
Link Kaggle input folders into the repo's expected `dataset/` structure.

In [ ]:
!rm -rf dataset
!mkdir -p dataset
!ln -s /kaggle/input/datasets/supratimghosh01/cufs-dataset-clean-stylized/dataset/photos dataset/photos
!ln -s /kaggle/input/datasets/supratimghosh01/cufs-dataset-clean-stylized/dataset/sketches dataset/sketches
!ln -s /kaggle/input/datasets/supratimghosh01/cufs-dataset-clean-stylized/dataset/stylized dataset/stylized
!ls -la dataset/

print(len(os.listdir('dataset/photos')))
print(len(os.listdir('dataset/sketches')))
print(len(os.listdir('dataset/stylized')))

## 6. Start Phase 2 Training

In [ ]:
!python phase2/training/train.py --epochs 200

## 7. Inspect Training Outputs
Display the latest generated sample grid.

In [ ]:
import glob
from IPython.display import display, Image as IPImage

sample_files = sorted(glob.glob('samples/phase2/epoch_*.png'))
if sample_files:
    latest = sample_files[-1]
    print(f'Showing: {latest}')
    display(IPImage(filename=latest, width=900))
else:
    print('No sample images found — training may not have completed.')

## 8. Quick Inference Preview

In [ ]:
import os
from IPython.display import display, Image as IPImage

first_sketch = sorted(os.listdir('dataset/sketches'))[0]
input_path = f'dataset/sketches/{first_sketch}'

!python phase2/inference/generate_color.py --input $input_path --output phase2_preview.png --checkpoint phase2/checkpoints/best_generator.pth
display(IPImage(filename='phase2_preview.png', width=600))

## 9. Package Outputs for Download
Zip best checkpoint and sample grids so files appear in Kaggle output.

In [ ]:
!zip -r /kaggle/working/deepsketch_phase2_artifacts.zip phase2/checkpoints samples/phase2 phase2_preview.png
print('Outputs zipped to /kaggle/working/deepsketch_phase2_artifacts.zip')